# Assignment 3: Plant Disease Detection - Fast Version
## Simplified CNN with Reduced Configurations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns

In [ ]:
# Create output directory
os.makedirs('outputs', exist_ok=True)
print(f"Output directory: {os.path.abspath('outputs')}")

In [ ]:
# Dataset paths - UPDATE THIS!
dataset_base = r'New Plant Diseases Dataset(Augmented)'
train_dir = os.path.join(dataset_base, 'train')
valid_dir = os.path.join(dataset_base, 'valid')  # Try 'valid' first

# Check if valid folder exists, if not try other names
if not os.path.exists(valid_dir):
    # Try alternative names
    for alt_name in ['validation', 'val', 'test']:
        alt_path = os.path.join(dataset_base, alt_name)
        if os.path.exists(alt_path):
            valid_dir = alt_path
            print(f"Using '{alt_name}' folder for validation")
            break
    else:
        # No separate validation folder found, will split from training
        valid_dir = None
        print("No separate validation folder found. Will split from training data (80/20).")
else:
    print(f"Found validation folder: {valid_dir}")

# REDUCED image size for MUCH faster training
img_size = 64  # Much smaller = much faster
batch_size = 64  # Larger batch = fewer iterations

print(f"\nImage size: {img_size}x{img_size}")
print(f"Batch size: {batch_size}")

In [ ]:
# Data generators
if valid_dir is None:
    # No separate validation folder - split from training
    print("\nUsing 80/20 split from training data")
    train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
    valid_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        subset='training',
        shuffle=True
    )
    
    valid_generator = valid_datagen.flow_from_directory(
        train_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        subset='validation',
        shuffle=False
    )
else:
    # Separate validation folder exists
    print("\nUsing separate validation folder")
    train_datagen = ImageDataGenerator(rescale=1./255)
    valid_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=True
    )
    
    valid_generator = valid_datagen.flow_from_directory(
        valid_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )

num_classes = len(train_generator.class_indices)
class_names = list(train_generator.class_indices.keys())

print(f"\nNumber of classes: {num_classes}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {valid_generator.samples}")
print(f"\nFirst 5 classes: {class_names[:5]}")

In [ ]:
# ONLY 3 CONFIGURATIONS for speed
configs = [
    {'name': 'ReLU, Batch 64', 'activation': 'relu', 'batch_size': 64},
    {'name': 'ReLU, Batch 128', 'activation': 'relu', 'batch_size': 128},
    {'name': 'Tanh, Batch 64', 'activation': 'tanh', 'batch_size': 64},
]

results = []
histories = []

print("Training 3 configurations (this will take ~20-30 min total)\n")

In [ ]:
# Train models
for i, config in enumerate(configs, 1):
    print(f"\n{'='*70}")
    print(f"Config {i}/3: {config['name']}")
    print(f"{'='*70}")
    
    # Update batch size
    train_generator.batch_size = config['batch_size']
    valid_generator.batch_size = config['batch_size']
    
    # Simple CNN model
    model = Sequential([
        Conv2D(32, 3, activation=config['activation'], input_shape=(img_size, img_size, 3)),
        MaxPooling2D(2),
        Conv2D(64, 3, activation=config['activation']),
        MaxPooling2D(2),
        Flatten(),
        Dense(128, activation=config['activation']),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Train for ONLY 3 epochs (faster)
    print("Training...")
    history = model.fit(
        train_generator,
        epochs=3,  # Very few epochs
        validation_data=valid_generator,
        verbose=1
    )
    
    # Get final accuracy
    val_acc = history.history['val_accuracy'][-1]
    val_loss = history.history['val_loss'][-1]
    
    results.append({
        'config': config['name'],
        'activation': config['activation'],
        'batch_size': config['batch_size'],
        'val_accuracy': val_acc,
        'val_loss': val_loss,
        'epochs': 3
    })
    
    histories.append(history)
    
    print(f"✓ Validation Accuracy: {val_acc:.4f}")
    print(f"✓ Validation Loss: {val_loss:.4f}")

print("\n" + "="*70)
print("ALL TRAINING COMPLETE!")
print("="*70)

In [ ]:
# Results table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('val_accuracy', ascending=False)
print("\nRESULTS:")
print(results_df)
results_df.to_csv('outputs/assignment3_results.csv', index=False)
print("\nSaved: outputs/assignment3_results.csv")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].barh(results_df['config'], results_df['val_accuracy'], color='skyblue', edgecolor='black')
axes[0].set_xlabel('Validation Accuracy', fontweight='bold')
axes[0].set_title('Accuracy Comparison', fontweight='bold')
axes[0].set_xlim(0, 1)
for i, v in enumerate(results_df['val_accuracy']):
    axes[0].text(v, i, f' {v:.3f}', va='center')

# Loss comparison
axes[1].barh(results_df['config'], results_df['val_loss'], color='salmon', edgecolor='black')
axes[1].set_xlabel('Validation Loss', fontweight='bold')
axes[1].set_title('Loss Comparison', fontweight='bold')
for i, v in enumerate(results_df['val_loss']):
    axes[1].text(v, i, f' {v:.3f}', va='center')

plt.tight_layout()
plt.savefig('outputs/assignment3_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: outputs/assignment3_comparison.png")

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (history, config) in enumerate(zip(histories, configs)):
    ax = axes[idx]
    epochs_range = range(1, len(history.history['accuracy']) + 1)
    
    ax.plot(epochs_range, history.history['accuracy'], 'b-', label='Train Acc', linewidth=2)
    ax.plot(epochs_range, history.history['val_accuracy'], 'r-', label='Val Acc', linewidth=2)
    ax.set_xlabel('Epoch', fontweight='bold')
    ax.set_ylabel('Accuracy', fontweight='bold')
    ax.set_title(config['name'], fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/assignment3_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: outputs/assignment3_training_curves.png")

In [ ]:
# Best model summary
best_idx = results_df['val_accuracy'].idxmax()
best_result = results_df.loc[best_idx]

print("\n" + "="*70)
print("BEST MODEL")
print("="*70)
print(f"Configuration: {best_result['config']}")
print(f"Activation: {best_result['activation']}")
print(f"Batch Size: {best_result['batch_size']}")
print(f"Validation Accuracy: {best_result['val_accuracy']:.4f} ({best_result['val_accuracy']*100:.2f}%)")
print(f"Validation Loss: {best_result['val_loss']:.4f}")
print(f"Epochs Trained: {best_result['epochs']}")
print("="*70)